4 products and 10 data points

|    | s | T    |
|----|---|------|
| 1  | a | abcd |
| 2  | b | abcd |
| 3  | c | abc  |
| 4  | a | abc  |
| 5  | b | ab   |
| 6  | c | bc   |
| 7  | b | ab   |
| 8  | a | ac   |
| 9  | c | ac   |
| 10 | b | bc   |

In [6]:
using Pkg
Pkg.add("Ipopt")

   Resolving package versions...
   Installed Hwloc_jll ───────────── v2.12.2+0
   Installed SPRAL_jll ───────────── v2025.5.20+0
   Installed Ipopt_jll ───────────── v300.1400.1900+0
   Installed METIS_jll ───────────── v5.1.3+0
   Installed OpenBLAS32_jll ──────── v0.3.29+0
   Installed MUMPS_seq_jll ───────── v500.800.100+0
   Installed Xorg_libpciaccess_jll ─ v0.18.1+0
   Installed XML2_jll ────────────── v2.13.8+0
   Installed Ipopt ───────────────── v1.11.0
    Updating `D:\Julia\packages\environments\v1.11\Project.toml`
  [b6b21f68] + Ipopt v1.11.0
    Updating `D:\Julia\packages\environments\v1.11\Manifest.toml`
  [b6b21f68] + Ipopt v1.11.0
  [ae81ac8f] + ASL_jll v0.1.3+0
  [e33a78d0] + Hwloc_jll v2.12.2+0
  [9cc047cb] + Ipopt_jll v300.1400.1900+0
  [d00139f3] + METIS_jll v5.1.3+0
  [d7ed1dd3] + MUMPS_seq_jll v500.800.100+0
  [656ef2d0] + OpenBLAS32_jll v0.3.29+0
  [319450e9] + SPRAL_jll v2025.5.20+0
⌅ [02c8fc9c] + XML2_jll v2.13.8+0
  [a65dc6b1] + Xorg_libpciaccess_jll v0.18.1

In [2]:
using Statistics, Combinatorics, JuMP, LinearAlgebra
import Ipopt

In [3]:
data = [
    ([1], [1,1,1,1]),
    ([2], [1,1,1,1]),
    ([3], [1,1,1,0]),
    ([1], [1,1,1,0]),
    ([2], [1,1,0,0]),
    ([3], [0,1,1,0]),
    ([2], [1,1,0,0]),
    ([1], [1,0,1,0]),
    ([3], [1,0,1,0]),
    ([2], [0,1,1,0])
]

10-element Vector{Tuple{Vector{Int64}, Vector{Int64}}}:
 ([1], [1, 1, 1, 1])
 ([2], [1, 1, 1, 1])
 ([3], [1, 1, 1, 0])
 ([1], [1, 1, 1, 0])
 ([2], [1, 1, 0, 0])
 ([3], [0, 1, 1, 0])
 ([2], [1, 1, 0, 0])
 ([1], [1, 0, 1, 0])
 ([3], [1, 0, 1, 0])
 ([2], [0, 1, 1, 0])

In [22]:
function kpartitions(arr, k)
    n = length(arr)
    res = []
    
    T = eltype(arr)
    
    function backtrack(i, parts)
        if i > n
            push!(res, deepcopy(parts))
            return
        end
        for j in 1:k
            push!(parts[j], arr[i])
            backtrack(i+1, parts)
            pop!(parts[j])
        end
    end

    backtrack(1, [Vector{T}() for _ in 1:k])
    return collect(res)
end

kpartitions(data,2)[1]

2-element Vector{Vector{Tuple{Vector{Int64}, Vector{Int64}}}}:
 [([1], [1, 1, 1, 1]), ([2], [1, 1, 1, 1]), ([3], [1, 1, 1, 0]), ([1], [1, 1, 1, 0]), ([2], [1, 1, 0, 0]), ([3], [0, 1, 1, 0]), ([2], [1, 1, 0, 0]), ([1], [1, 0, 1, 0]), ([3], [1, 0, 1, 0]), ([2], [0, 1, 1, 0])]
 []

In [31]:
function fit_mnl(data::Array)
    if isempty(data)
        return [], 0.0
    end
    
    n = length(data[1][2])

    model = Model(Ipopt.Optimizer)
    set_silent(model)

    @variable(model, u[1:n])
    @NLobjective(model, Max,
        sum(u[d[1][1]] - log(sum(d[2][j] * exp(u[j]) for j in 1:n)) for d in data)
    )
    optimize!(model)
    return value.(u), objective_value(model)

end

fit_mnl (generic function with 1 method)

In [32]:
function partalg(data::Array, num_part = 2)
    partitions = kpartitions(data, num_part)
    max_part = partitions[1]
    max_logp = sum(fit_mnl(p)[2] for p in partitions[1])
    
    for part in partitions
        logp = sum(fit_mnl(p)[2] for p in part)
        if logp > max_logp
            max_logp = logp 
            max_part = part
        end
    end
    return max_part, max_logp
end

partalg (generic function with 2 methods)

In [33]:
partalg(data)

([[([1], [1, 1, 1, 1]), ([1], [1, 1, 1, 0]), ([3], [0, 1, 1, 0]), ([1], [1, 0, 1, 0])], [([2], [1, 1, 1, 1]), ([3], [1, 1, 1, 0]), ([2], [1, 1, 0, 0]), ([2], [1, 1, 0, 0]), ([3], [1, 0, 1, 0]), ([2], [0, 1, 1, 0])]], -1.9095425204000618)

In [34]:
fit_mnl(data)

([-7.671759940823536, -7.325186350494057, -7.671759940823536, -25.8828887586258], -8.437283064499717)

In [65]:
function rum(s, T::Vector{Int}, λ, perms)

    s_int = isa(s, Integer) ? s : (length(s) == 1 ? s[1] : error("s must be single element"))

    choice_set = findall(x -> x == 1, T)
    n = length(perms[1])
    # check
    if length(T) != n
        error("T must be length n = $n")
    end
    # treat nonzero as included (works for Int/Bool/Float)
    if isempty(choice_set) #empty set
        @warn "empty choice set T: returning 0.0"
        return 0.0
    end
    if !(s_int in choice_set) #outside option
        return 0.0
    end

    prob = 0.0
    for (i, perm) in enumerate(perms)
        if all(s_int in perm[1:findfirst(==(t), perm)] for t in choice_set if t != s_int)
            prob += λ[i]
        end
    end
    return prob
end

rum (generic function with 2 methods)

In [66]:
λ = []
perms = collect(permutations(1:4))
for i in perms
    push!(λ,1/length(perms))
end

rum(1,[1,1,0,1],λ,perms)

0.3333333333333333

In [78]:
function fit_rum(data::Array)
    if isempty(data)
        return [], 0.0
    end
    
    n = length(data[1][2])
    perms = collect(permutations(1:n))
    m = length(perms)

    # A collect the indices of permutation that satisfies (s, choice_set) 
    A = Dict()
    for (idx, (s, choice_set_unprocessed)) in enumerate(data)
        choice_set = findall(x -> x == 1, choice_set_unprocessed) 
        A[idx] = [ (all(s in perm[1:findfirst(==(t), perm)] 
                        for t in choice_set if t != s)) ? 1.0 : 0.0
                   for perm in perms ]
    end

    model = Model(Ipopt.Optimizer)
    set_silent(model)

    @variable(model, λ[1:m] >= 0)
    @constraint(model, sum(λ) == 1)

    @NLobjective(model, Max,
        sum(log(sum(λ[j] * A[idx][j] for j in 1:m)) for idx in 1:length(data))
    )

    optimize!(model)
    return value.(λ), objective_value(model)
end

fit_rum (generic function with 2 methods)

In [81]:
function fit_rum(data::Array)
    if isempty(data)
        return [], 0.0
    end
    
    n = length(data[1][2])
    perms = collect(permutations(1:n))
    model = Model(Ipopt.Optimizer)
    set_silent(model)

    @variable(model, λ[1:length(perms)])
    register(model, :rum, 4, rum; autodiff = true)
    @NLobjective(model, Max,
        sum(log(rum(d[1], d[2], λ, perms)) for d in data)
    )
    @constraint(model, c, sum(λ)==1)
    optimize!(model)
    return objective_value(model)
end

fit_rum (generic function with 2 methods)

In [82]:
fit_rum(data)

-7.9779678733823935